# Step 17 — Inference, 5-model 224px ensemble

**Settings: `GPU T4 x2`, `Internet OFF`.**

**Inputs:** competition data + your `rsna-knee-checkpoints` dataset

## About that "broken" dataset

Kaggle unzipped your `.pt` files because PyTorch checkpoints *are* zip archives internally. Nothing was lost — the parts are all there (`data.pkl`, `byteorder`, `data/`, `.data/`), just unpacked into directories.

Cell 2 re-zips them back into loadable `.pt` files. The round-trip is byte-identical, so this is exactly equivalent to having uploaded them properly.

## Preprocessing must match the 224px cache

`PX=224`, `NSL=32`, and **laterality resolved for every study** — the tag when present, otherwise the geometric rule (`x_mean > -37.5` implies left) that agreed with ground truth on 97.5% of tagged studies.

That last part matters: the earlier caches skipped mirroring whenever the tag was missing, which left ~24% of studies with medial and lateral reversed. The models you're loading were trained on correctly mirrored data, so inference must mirror the same way.

## Expected

Gold holdout: 3 of these models ensembled to **0.8350** (vs 0.825 for five 160px models, which scored **0.856** on the leaderboard). With five models and the usual +0.031 offset, expect roughly **0.869**.

Runtime ~70 min against a 9-hour limit.

### 1 · Setup

In [ ]:
import os, gc, glob, time, zipfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom, torch, torch.nn as nn
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
PX, NSL, NSER = 224, 32, 3
LAT_THR, LAT_SIGN = -37.5, 1          # validated geometric rule

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
test = pd.read_csv(f"{ROOT}/test.csv")
tser = pd.read_csv(f"{ROOT}/test_series.csv")
studies = test["StudyInstanceUID"].tolist()
print(f"test studies: {len(studies):,}")

### 2 · Rebuild the checkpoints

Finds every `data.pkl` under the dataset and re-zips its parent directory into a valid `.pt`.

In [ ]:
DS = glob.glob("/kaggle/input/*checkpoint*") + glob.glob("/kaggle/input/*rsna-knee-check*")
DS = DS[0] if DS else None
print("dataset root:", DS)

def rebuild(src_dir, out_path):
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_STORED) as z:
        for r, d, files in os.walk(src_dir):
            for name in files:
                full = os.path.join(r, name)
                rel  = os.path.relpath(full, src_dir).replace("\\", "/")
                z.write(full, "archive/" + rel)
    return out_path

CKPTS = []
for pkl in sorted(glob.glob(f"{DS}/**/data.pkl", recursive=True)):
    src = os.path.dirname(pkl)
    tag = [p for p in src.split("/") if "fold" in p.lower()]
    name = (tag[-1] if tag else f"ck{len(CKPTS)}") + ".pt"
    out = rebuild(src, f"/kaggle/working/{name}")
    CKPTS.append(out)
    print(f"  rebuilt {name}  ({os.path.getsize(out)/1e6:.1f} MB)")

print(f"\n{len(CKPTS)} checkpoints rebuilt")
assert CKPTS, "no checkpoints found - check the dataset is attached"

### 3 · Model

In [ ]:
import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        return (x * a.softmax(-1).unsqueeze(-1)).sum(1)

class Net(nn.Module):
    def __init__(self, backbone="efficientnet_b0", nlab=12, hid=256, pretrained=False):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))
    def forward(self, v, mask):                       # v: (B, S, T, H, W)
        B, S, T, H, W = v.shape
        lo = torch.cat([v[:, :, :1],  v[:, :, :-1]], 2)
        hi = torch.cat([v[:, :, 1:],  v[:, :, -1:]], 2)
        x  = torch.stack([lo, v, hi], 3).reshape(B * S * T, 3, H, W)
        f = self.enc(x).reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        return self.head(self.ser_pool(f, mask))

models = []
for p in CKPTS:
    ck = torch.load(p, map_location="cpu", weights_only=False)
    m = Net(pretrained=False).cuda(); m.load_state_dict(ck["model"]); m.eval()
    models.append(m)
    print(f"fold {ck['fold']}: val {ck['val_auc']:.4f} gold {ck['gold_auc']:.4f} "
          f"px {ck.get('px','?')} nsl {ck.get('nsl','?')}")
print(f"\nensembling {len(models)} models")

### 4 · Decoder — matches the 224px laterality-fixed cache exactly

In [ ]:
def pick(g):
    out = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

chosen = {suid: pick(g) for suid, g in tser.groupby("StudyInstanceUID")}


def load_series(dirpath):
    files = os.listdir(dirpath)
    if not files: return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas: return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]; r, c = iop[:3], iop[3:]; n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos); metas = [metas[i] for i in order]
    take = np.linspace(0, len(metas)-1, NSL).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((PX, PX), np.float32)
        imgs.append(cv2.resize(a, (PX, PX), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0: vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    # laterality: tag first, then the validated geometric rule -- never skipped
    tag = next((m[4] for m in metas if m[4]), "")
    if tag:
        side = tag.strip().upper()[0]
    else:
        xs = [float(m[2][0]) for m in metas if m[2] is not None]
        side = ("L" if (np.mean(xs) - LAT_THR) * LAT_SIGN > 0 else "R") if xs else "R"
    if side == "L":
        vol = np.flip(vol, axis=int(np.argmax([abs(v[0]) for v in axes])))

    return (vol * 255).astype(np.uint8)


def load_study(i):
    suid = studies[i]
    out  = np.zeros((NSER, NSL, PX, PX), np.uint8)
    pres = np.zeros(NSER, np.float32)
    for k, cs in enumerate(chosen.get(suid, [None]*3)):
        if cs is None: continue
        p = f"{ROOT}/test_series/{suid}/{cs}"
        if not os.path.isdir(p): continue
        try:
            v = load_series(p)
            if v is not None: out[k] = v; pres[k] = 1.0
        except Exception:
            pass
    return i, out, pres

print("decoder ready")

### 5 · Decode and predict

In [ ]:
CHUNK, BS = 64, 3
preds = np.full((len(studies), len(LABELS)), 0.5, np.float32)

t0 = time.time()
for c0 in range(0, len(studies), CHUNK):
    idx = list(range(c0, min(c0 + CHUNK, len(studies))))
    with Pool(4) as pool:
        res = pool.map(load_study, idx)

    vols = np.stack([r[1] for r in res]).astype(np.float32) / 255.0
    mask = np.stack([r[2] for r in res])
    ids  = [r[0] for r in res]

    with torch.no_grad():
        for b0 in range(0, len(ids), BS):
            vb = torch.from_numpy(vols[b0:b0+BS]).cuda()
            mb = torch.from_numpy(mask[b0:b0+BS]).cuda()
            acc = None
            for m in models:
                with torch.amp.autocast("cuda"):
                    o = m(vb, mb).float().sigmoid()
                acc = o if acc is None else acc + o
            preds[ids[b0:b0+BS]] = (acc / len(models)).cpu().numpy()

    del vols; gc.collect()
    done = c0 + len(idx); el = time.time() - t0
    print(f"  {done}/{len(studies)}  {el/done:.2f}s/study  eta {(len(studies)-done)*el/done/60:.0f} min", flush=True)

print(f"\ndone in {(time.time()-t0)/60:.1f} min")

### 6 · Write submission

In [ ]:
sub = pd.DataFrame(preds, columns=LABELS)
sub.insert(0, "StudyInstanceUID", studies)
sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.head())
print(f"\nrows left at default 0.5: {(preds == 0.5).all(1).sum()}")